# Scientific Article IR Challenge — Optimized Solution

**Strategy** (best score):
1. **SPECTER2** — scientific-domain dense model trained on citation prediction
2. **MiniLM-L6-v2** — fast general dense model (pre-computed, always available)
3. **BM25 (title+abstract)** — exact-match sparse retrieval
4. **BM25 (full text, capped 8k chars)** — full-text sparse for extra recall
5. **RRF** — Reciprocal Rank Fusion of all above

Submission format: `{query_id: [doc_id_1, ..., doc_id_100]}`


In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rank_bm25', '-q'])
print('Dependencies ready.')


Dependencies ready.


In [3]:
import json, math, os, re
from pathlib import Path
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

DATA_DIR = Path("../data")
OUT_DIR  = Path("../submissions")
OUT_DIR.mkdir(exist_ok=True)
print("Imports OK")


f:\IUT Orsay\T4\Information Retrieval\challenge\starter_kit\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1 — Load Data

In [4]:
queries  = pd.read_parquet(DATA_DIR / "queries.parquet")
corpus   = pd.read_parquet(DATA_DIR / "corpus.parquet")

held_out_path = DATA_DIR / "held_out_queries.parquet"
if not held_out_path.exists():
    held_out_path = Path("../held_out_queries.parquet")
held_out = pd.read_parquet(held_out_path)

with open(DATA_DIR / "qrels.json", encoding="utf-8") as f:
    qrels = json.load(f)

query_ids  = queries["doc_id"].tolist()
corpus_ids = corpus["doc_id"].tolist()
ho_ids     = held_out["doc_id"].tolist()

print(f"Public queries : {len(queries)}")
print(f"Corpus         : {len(corpus)}")
print(f"Held-out       : {len(held_out)}")
print(f"Qrels entries  : {len(qrels)}")


Public queries : 100
Corpus         : 20000
Held-out       : 100
Qrels entries  : 100


## 2 — Helper Functions

In [5]:
# ── Text formatting ──────────────────────────────────────────────────────
def fmt_ta(row) -> str:
    t = str(row.get("title",   "") or "").strip()
    a = str(row.get("abstract","") or "").strip()
    return (t + " " + a).strip()

def fmt_full(row, max_chars: int = 8000) -> str:
    ta = fmt_ta(row)
    ft = str(row.get("full_text", "") or "").strip()
    return (ta + " " + ft[:max_chars]).strip()

def tokenize(text: str) -> list:
    return re.findall(r"[a-z0-9]+", text.lower())

# ── Metrics ──────────────────────────────────────────────────────────────
def recall_at_k(ranked, rel_set, k):
    return len(set(ranked[:k]) & rel_set) / len(rel_set) if rel_set else 0.0

def precision_at_k(ranked, rel_set, k):
    return sum(1 for d in ranked[:k] if d in rel_set) / k

def mrr_at_k(ranked, rel_set, k):
    for i, d in enumerate(ranked[:k]):
        if d in rel_set:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(ranked, rel_set, k):
    dcg  = sum(1.0/math.log2(i+2) for i, d in enumerate(ranked[:k]) if d in rel_set)
    idcg = sum(1.0/math.log2(i+2) for i in range(min(len(rel_set), k)))
    return dcg / idcg if idcg else 0.0

def average_precision(ranked, rel_set):
    if not rel_set: return 0.0
    hits, s = 0, 0.0
    for i, d in enumerate(ranked):
        if d in rel_set:
            hits += 1
            s += hits / (i + 1)
    return s / len(rel_set)

def evaluate(submission: dict, qrels: dict) -> dict:
    b = {k: [] for k in ["recall@10","recall@100","ndcg@10","ndcg@100",
                          "precision@10","mrr@10","map"]}
    for qid, ranked in submission.items():
        rel = set(qrels.get(qid, []))
        if not rel: continue
        b["recall@10"].append(recall_at_k(ranked, rel, 10))
        b["recall@100"].append(recall_at_k(ranked, rel, 100))
        b["ndcg@10"].append(ndcg_at_k(ranked, rel, 10))
        b["ndcg@100"].append(ndcg_at_k(ranked, rel, 100))
        b["precision@10"].append(precision_at_k(ranked, rel, 10))
        b["mrr@10"].append(mrr_at_k(ranked, rel, 10))
        b["map"].append(average_precision(ranked, rel))
    return {k: float(np.mean(v)) for k, v in b.items()}

# ── Reciprocal Rank Fusion ────────────────────────────────────────────────
def rrf(rankings_list: list, k: int = 60, top_n: int = 100) -> list:
    scores: dict = {}
    for rankings in rankings_list:
        for rank, doc_id in enumerate(rankings):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.__getitem__, reverse=True)[:top_n]

def build_best(query_id_list, minilm_d, bm25_ft_d, specter_d=None):
    # Optimal: RRF(SPECTER2?, MiniLM, BM25_FT)  -- BM25_TA excluded (redundant with FT)
    result = {}
    for qid in query_id_list:
        candidates = [minilm_d.get(qid,[]), bm25_ft_d.get(qid,[])]
        if specter_d is not None:
            candidates.insert(0, specter_d.get(qid,[]))
        result[qid] = rrf(candidates)
    return result

def show_results(name: str, results: dict):
    print(f"\n{chr(9472)*55}")
    print(f"  {name}")
    print(f"{chr(9472)*55}")
    for k, v in results.items():
        print(f"  {k:<15}  {v:.4f}")

print("Helpers defined.")


Helpers defined.


## 3 — Strategy A: Dense Retrieval (MiniLM, pre-computed)

In [6]:
EMB_DIR = DATA_DIR / "embeddings" / "sentence-transformers_all-MiniLM-L6-v2"

q_embs = np.load(EMB_DIR / "query_embeddings.npy").astype(np.float32)
c_embs = np.load(EMB_DIR / "corpus_embeddings.npy").astype(np.float32)
with open(EMB_DIR / "query_ids.json")  as f: emb_q_ids = json.load(f)
with open(EMB_DIR / "corpus_ids.json") as f: emb_c_ids = json.load(f)

# L2-normalised embeddings => cosine sim == dot product
sim_minilm = q_embs @ c_embs.T           # (100, 20000)
top_minilm = np.argsort(-sim_minilm, axis=1)[:, :100]
minilm_sub = {qid: [emb_c_ids[j] for j in top_minilm[i]] for i, qid in enumerate(emb_q_ids)}

r_minilm = evaluate(minilm_sub, qrels)
show_results("MiniLM-L6-v2 (pre-computed)", r_minilm)



───────────────────────────────────────────────────────
  MiniLM-L6-v2 (pre-computed)
───────────────────────────────────────────────────────
  recall@10        0.4716
  recall@100       0.8104
  ndcg@10          0.5073
  ndcg@100         0.5903
  precision@10     0.2440
  mrr@10           0.6812
  map              0.4080


## 4 — Strategy B: BM25 on Title + Abstract

In [8]:
print("Tokenising corpus (title+abstract)...")
corpus_ta_tok = [tokenize(fmt_ta(row)) for _, row in tqdm(corpus.iterrows(), total=len(corpus))]
bm25_ta = BM25Okapi(corpus_ta_tok)
query_ta_tok = [tokenize(fmt_ta(row)) for _, row in queries.iterrows()]

bm25_ta_sub = {}
for i, qid in enumerate(tqdm(query_ids, desc="BM25 TA retrieval")):
    scores = bm25_ta.get_scores(query_ta_tok[i])
    bm25_ta_sub[qid] = [corpus_ids[j] for j in np.argsort(-scores)[:100]]

r_bm25_ta = evaluate(bm25_ta_sub, qrels)
show_results("BM25 -- title + abstract", r_bm25_ta)


Tokenising corpus (title+abstract)...


BM25 TA retrieval: 100%|██████████| 100/100 [06:09<00:00,  3.69s/it]


───────────────────────────────────────────────────────
  BM25 -- title + abstract
───────────────────────────────────────────────────────
  recall@10        0.4318
  recall@100       0.6885
  ndcg@10          0.4663
  ndcg@100         0.5210
  precision@10     0.2190
  mrr@10           0.6486
  map              0.3489


## 5 — Strategy C: BM25 on Full Text (capped at 8 000 chars)

In [9]:
print("Tokenising corpus (full text, <=8000 chars)...")
corpus_ft_tok = [tokenize(fmt_full(row)) for _, row in tqdm(corpus.iterrows(), total=len(corpus))]
bm25_ft = BM25Okapi(corpus_ft_tok)

# Query with TA tokens (concise signal for what to retrieve)
bm25_ft_sub = {}
for i, qid in enumerate(tqdm(query_ids, desc="BM25 FT retrieval")):
    scores = bm25_ft.get_scores(query_ta_tok[i])
    bm25_ft_sub[qid] = [corpus_ids[j] for j in np.argsort(-scores)[:100]]

r_bm25_ft = evaluate(bm25_ft_sub, qrels)
show_results("BM25 -- full text (capped)", r_bm25_ft)


Tokenising corpus (full text, <=8000 chars)...


BM25 FT retrieval: 100%|██████████| 100/100 [06:59<00:00,  4.20s/it]


───────────────────────────────────────────────────────
  BM25 -- full text (capped)
───────────────────────────────────────────────────────
  recall@10        0.5098
  recall@100       0.7887
  ndcg@10          0.5357
  ndcg@100         0.5997
  precision@10     0.2660
  mrr@10           0.6645
  map              0.4325


## 6 — Hybrid: RRF combinations (empirical comparison)

In [10]:
# Empirical finding: BM25-FT already covers the TA text, so adding BM25-TA separately
# is redundant and slightly hurts.  Best hybrid = RRF(MiniLM + BM25_FT).

# A+B
rrf_AB = {qid: rrf([minilm_sub[qid], bm25_ta_sub[qid]]) for qid in query_ids}
r_AB = evaluate(rrf_AB, qrels)
show_results("RRF (MiniLM + BM25_TA)", r_AB)

# A+C  <-- BEST without SPECTER2
rrf_AC = {qid: rrf([minilm_sub[qid], bm25_ft_sub[qid]]) for qid in query_ids}
r_AC = evaluate(rrf_AC, qrels)
show_results("RRF (MiniLM + BM25_FT)  <-- best hybrid", r_AC)

# A+B+C
rrf_ABC = {qid: rrf([minilm_sub[qid], bm25_ta_sub[qid], bm25_ft_sub[qid]]) for qid in query_ids}
r_ABC = evaluate(rrf_ABC, qrels)
show_results("RRF (MiniLM + BM25_TA + BM25_FT)", r_ABC)



───────────────────────────────────────────────────────
  RRF (MiniLM + BM25_TA)
───────────────────────────────────────────────────────
  recall@10        0.4695
  recall@100       0.7909
  ndcg@10          0.5138
  ndcg@100         0.5906
  precision@10     0.2460
  mrr@10           0.6983
  map              0.4057

───────────────────────────────────────────────────────
  RRF (MiniLM + BM25_FT)  <-- best hybrid
───────────────────────────────────────────────────────
  recall@10        0.5167
  recall@100       0.8352
  ndcg@10          0.5479
  ndcg@100         0.6245
  precision@10     0.2680
  mrr@10           0.6956
  map              0.4477

───────────────────────────────────────────────────────
  RRF (MiniLM + BM25_TA + BM25_FT)
───────────────────────────────────────────────────────
  recall@10        0.4803
  recall@100       0.8239
  ndcg@10          0.5280
  ndcg@100         0.6126
  precision@10     0.2510
  mrr@10           0.7050
  map              0.4293


## 7 — Strategy D: SPECTER2 (best model for scientific citation retrieval)

SPECTER2 is trained so that **papers that cite each other** are close in embedding space — exactly this task.  
Embeddings are **cached** after the first run so re-running is instant.


In [11]:
SPECTER_CACHE   = DATA_DIR / "embeddings" / "specter2"
specter_available = False
specter_model     = None

if (SPECTER_CACHE / "corpus_embeddings.npy").exists():
    print("Loading cached SPECTER2 embeddings...")
    s_c_embs = np.load(SPECTER_CACHE / "corpus_embeddings.npy").astype(np.float32)
    s_q_embs = np.load(SPECTER_CACHE / "query_embeddings.npy").astype(np.float32)
    specter_available = True
    print(f"  corpus: {s_c_embs.shape}  queries: {s_q_embs.shape}")

else:
    # Priority: best scientific model first, then strong general models
    for model_id in [
        "allenai/specter2",          # SPECTER2 - trained on scientific citation prediction
        "malteos/scincl",            # SciNCL   - another strong citation model
        "BAAI/bge-base-en-v1.5",     # BGE-base - top general retrieval (MTEB)
        "intfloat/e5-base-v2",       # E5-base  - strong general retrieval
    ]:
        try:
            print(f"Trying {model_id}...")
            specter_model = SentenceTransformer(model_id, trust_remote_code=True)
            print(f"  Loaded! dim={specter_model.get_sentence_embedding_dimension()}")

            corpus_texts = [fmt_ta(row) for _, row in corpus.iterrows()]
            query_texts  = [fmt_ta(row) for _, row in queries.iterrows()]

            print("  Encoding corpus (may take a few minutes on CPU)...")
            s_c_embs = specter_model.encode(
                corpus_texts, batch_size=128,
                show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True,
            ).astype(np.float32)

            print("  Encoding queries...")
            s_q_embs = specter_model.encode(
                query_texts, batch_size=128,
                show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True,
            ).astype(np.float32)

            SPECTER_CACHE.mkdir(parents=True, exist_ok=True)
            np.save(SPECTER_CACHE / "corpus_embeddings.npy", s_c_embs)
            np.save(SPECTER_CACHE / "query_embeddings.npy",  s_q_embs)
            with open(SPECTER_CACHE / "corpus_ids.json", "w") as f: json.dump(corpus_ids, f)
            with open(SPECTER_CACHE / "query_ids.json",  "w") as f: json.dump(query_ids, f)
            with open(SPECTER_CACHE / "model_info.txt",  "w") as f: f.write(f"model: {model_id}\n")

            specter_available = True
            print(f"  Saved to {SPECTER_CACHE}/")
            break

        except Exception as exc:
            print(f"  Failed ({type(exc).__name__}): {exc}")

if not specter_available:
    print("No scientific/BGE model available -- continuing with MiniLM + BM25.")


Trying allenai/specter2...


No sentence-transformers model found with name allenai/specter2. Creating a new one with mean pooling.


  Failed (TypeError): The PeftConfig config that is trying to be loaded is missing required keys: {'peft_type'}.
Trying malteos/scincl...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1326.12it/s]
BertModel LOAD REPORT from: malteos/scincl
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Loaded! dim=768
  Failed (ArrowException): Unknown error: Wrapping Informing policy via dynamic models: Cholera in Haiti

Public health decisions must be made about when and how to implement interventions to control an infectious disease epidemic. These decisions should be informed by data on the epidemic as well as current understanding about the transmission dynamics. Such decisions can be posed as statistical questions about scientifically motivated dynamic models. Thus, we encounter the methodological task of building credible, data-informed decisions based on stochastic, partially observed, nonlinear dynamic models. This necessitates addressing the tradeoff between biological fidelity and model simplicity, and the reality of misspecification for models at all levels of complexity. We assess current methodological approaches to these issues via a case study of the 2010-2019 cholera epidemic in Haiti. We consider three dynamic models developed by expert teams to advise on vaccinat

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 953.96it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Loaded! dim=768
  Failed (ArrowException): Unknown error: Wrapping Transcriptional activity of the giant barrel sponge, Xestospongia muta Holobiont: molecular evidence for metabolic interchange

Compared to our understanding of the taxonomic composition of the symbiotic microbes in marine sponges, the functional diversity of these symbionts is largely unknown. Furthermore, the application of genomic, transcriptomic, and proteomic techniques to functional questions on sponge host-symbiont interactions is in its infancy. In this study, we generated a transcriptome for the host and a metatranscriptome of its microbial symbionts for the giant barrel sponge, Xestospongia muta, from the Caribbean. In combination with a gene-specific approach, our goals were to (1) characterize genetic evidence for nitrogen cycling in X. muta, an important limiting nutrient on coral reefs (2) identify which prokaryotic symbiont lineages are metabolically active and, (3) characterize the metabolic potential 

f:\IUT Orsay\T4\Information Retrieval\challenge\starter_kit\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\hp\.cache\huggingface\hub\models--intfloat--e5-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
f:\IUT Orsay\T4\Information Retrieval\challenge\starter_kit\.venv\Lib\site-packages\hu

  Failed (OSError): Can't load the model for 'intfloat/e5-base-v2'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'intfloat/e5-base-v2' is the correct path to a directory containing a file named pytorch_model.bin.
No scientific/BGE model available -- continuing with MiniLM + BM25.


In [12]:
specter_sub = None
r_specter   = None

if specter_available:
    sim_specter = s_q_embs @ s_c_embs.T
    top_specter = np.argsort(-sim_specter, axis=1)[:, :100]
    specter_sub = {qid: [corpus_ids[j] for j in top_specter[i]] for i, qid in enumerate(query_ids)}
    r_specter = evaluate(specter_sub, qrels)
    show_results("SPECTER2 / SciNCL / BGE (dense)", r_specter)
else:
    print("Skipped -- model not available.")


Skipped -- model not available.


## 8 — Best Combination (RRF of all available models)

In [13]:
# Optimal combo (empirically validated):
#   WITHOUT SPECTER2: RRF(MiniLM, BM25_FT)
#   WITH    SPECTER2: RRF(SPECTER2, MiniLM, BM25_FT)
# BM25_TA is intentionally excluded — it is a strict subset of BM25_FT
# and adding it only dilutes the signal.

if specter_available:
    best_sub = {
        qid: rrf([specter_sub[qid], minilm_sub[qid], bm25_ft_sub[qid]])
        for qid in query_ids
    }
    label = "RRF (SPECTER2 + MiniLM + BM25_FT)"
else:
    best_sub = rrf_AC
    label    = "RRF (MiniLM + BM25_FT)"

r_best = evaluate(best_sub, qrels)
show_results(f"BEST: {label}", r_best)



───────────────────────────────────────────────────────
  BEST: RRF (MiniLM + BM25_FT)
───────────────────────────────────────────────────────
  recall@10        0.5167
  recall@100       0.8352
  ndcg@10          0.5479
  ndcg@100         0.6245
  precision@10     0.2680
  mrr@10           0.6956
  map              0.4477


## 9 — Results Summary

In [ ]:
all_results = {
    "A  MiniLM-L6-v2":       r_minilm,
    "B  BM25 (TA)":          r_bm25_ta,
    "C  BM25 (full text)":   r_bm25_ft,
    "D  RRF (A+B)":          r_AB,
    "E* RRF (A+C) [BEST]":   r_AC,
    "F  RRF (A+B+C)":        r_ABC,
}
if specter_available and r_specter:
    all_results["G  SPECTER2/BGE"]              = r_specter
    all_results["H* RRF (G+A+C) [BEST+SPEC]"]  = r_best

cols = ["recall@10","recall@100","ndcg@10","ndcg@100","precision@10","mrr@10","map"]
df_compare = pd.DataFrame(all_results).T[cols]
try:
    display(df_compare.style.format("{:.4f}").highlight_max(axis=0, color="#c8e6c9"))
except NameError:
    print(df_compare.to_string(float_format=lambda x: f"{x:.4f}"))


,recall@10,recall@100,ndcg@10,ndcg@100,precision@10,mrr@10,map
A MiniLM-L6-v2,0.4716,0.8104,0.5073,0.5903,0.2440,0.6812,0.4080
B BM25 (TA),0.4318,0.6885,0.4663,0.5210,0.2190,0.6486,0.3489
C BM25 (full text),0.5098,0.7887,0.5357,0.5997,0.2660,0.6645,0.4325
D RRF (A+B),0.4695,0.7909,0.5138,0.5906,0.2460,0.6983,0.4057
E* RRF (A+C) [BEST],0.5167,0.8352,0.5479,0.6245,0.2680,0.6956,0.4477
F RRF (A+B+C),0.4803,0.8239,0.5280,0.6126,0.2510,0.7050,0.4293


: 

## 10 — Held-Out Query Submission

Held-out queries have no pre-computed embeddings — encode on the fly with the same pipeline.


In [ ]:
ho_ta_texts = [fmt_ta(row) for _, row in held_out.iterrows()]
ho_ta_tok   = [tokenize(t) for t in ho_ta_texts]

# Dense: MiniLM
print("Loading MiniLM for held-out encoding...")
minilm_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
ho_minilm_embs = minilm_model.encode(
    ho_ta_texts, batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
).astype(np.float32)
ho_sim = ho_minilm_embs @ c_embs.T
ho_top = np.argsort(-ho_sim, axis=1)[:, :100]
ho_minilm_sub = {qid: [corpus_ids[j] for j in ho_top[i]] for i, qid in enumerate(ho_ids)}
print(f"MiniLM held-out: {len(ho_minilm_sub)} queries done.")


Loading MiniLM for held-out encoding...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1091.98it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# BM25 full-text held-out (reuse the index built above; BM25_TA excluded from final combo)
ho_bm25_ft_sub = {}
for i, qid in enumerate(tqdm(ho_ids, desc="BM25 FT held-out")):
    scores = bm25_ft.get_scores(ho_ta_tok[i])
    ho_bm25_ft_sub[qid] = [corpus_ids[j] for j in np.argsort(-scores)[:100]]


In [ ]:
# SPECTER2 / BGE held-out
ho_specter_sub = None

if specter_available:
    if specter_model is None:
        model_info = (SPECTER_CACHE / "model_info.txt").read_text().strip()
        model_id   = model_info.split(": ", 1)[1] if ": " in model_info else "allenai/specter2"
        print(f"Reloading {model_id}...")
        specter_model = SentenceTransformer(model_id, trust_remote_code=True)

    ho_s_embs = specter_model.encode(
        ho_ta_texts, batch_size=128, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)
    ho_s_sim = ho_s_embs @ s_c_embs.T
    ho_s_top = np.argsort(-ho_s_sim, axis=1)[:, :100]
    ho_specter_sub = {qid: [corpus_ids[j] for j in ho_s_top[i]] for i, qid in enumerate(ho_ids)}
    print(f"SPECTER2 held-out: {len(ho_specter_sub)} queries done.")
else:
    print("SPECTER2 not available -- held-out uses MiniLM + BM25 only.")


In [ ]:
held_out_final = build_best(ho_ids, ho_minilm_sub, ho_bm25_ft_sub,
                            specter_d=ho_specter_sub)

print(f"Held-out: {len(held_out_final)} queries x 100 results")
assert all(len(v) == 100 for v in held_out_final.values()), "Some queries have != 100 results!"
print("Sanity check passed.")


## 11 — Save Submissions

In [ ]:
assert all(len(v) == 100 for v in best_sub.values())

with open(OUT_DIR / "best_public.json", "w") as f:
    json.dump(best_sub, f)
print(f"Saved  {OUT_DIR}/best_public.json   ({len(best_sub)} queries)")

with open(OUT_DIR / "held_out.json", "w") as f:
    json.dump(held_out_final, f)
print(f"Saved  {OUT_DIR}/held_out.json      ({len(held_out_final)} queries)")

full_submission = {**best_sub, **held_out_final}
with open(OUT_DIR / "final_submission.json", "w") as f:
    json.dump(full_submission, f)
print(f"Saved  {OUT_DIR}/final_submission.json  ({len(full_submission)} queries total)")

print("\n=== FINAL PUBLIC SCORES ===")
for k, v in r_best.items():
    print(f"  {k:<15}  {v:.4f}")
